# audioseal_robust -- debug run on free GPU (Colab/Kaggle)

Purpose: this notebook is **not** the real training run. It's a quick, cheap
correctness check -- clone the repo, install deps, pull a small checkpoint,
run a handful of training/eval steps on a free GPU -- to confirm the
`audioseal_robust` fine-tuning pipeline (generator-only, sampled
reconstruction attacks, see `src/audioseal_robust/`) actually runs correctly
end to end *before* asking for a real parallel training run on shared/main
compute. Once every cell below runs clean, the exact recipe/config used here
is what gets handed off for the real run -- don't scale this notebook up
in place.

Two attack backbones are supported via the `recipe=` mechanism
(`src/audioseal_robust/config/recipes.yaml`) -- swapping which diffusion
model the generator trains against is a config choice, not a code change:
- `recipe=sgmse` -- trains against SGMSE (OU-VE SDE speech enhancement)
- `recipe=diff_erase` -- trains against DiffErase/AudioLDM (latent-diffusion
  resynthesis) -- heavier download (~10GB), see the optional section below.


## 1. Clone the repo

Clones whatever branch is currently on GitHub -- make sure your local changes are pushed there first (this project has been using the `colab-debug` branch, set as the default below).

In [ ]:
#@title Clone repo
REPO_URL = "https://github.com/martysai/psiml11-audio-with-diffusion.git"  #@param {type:"string"}
BRANCH = "colab-debug"  #@param {type:"string"}

!git clone --branch $BRANCH $REPO_URL
%cd psiml11-audio-with-diffusion
!git log --oneline -5


## 2. Check GPU

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))


## 3. Install dependencies

`requirements-training.txt` covers the base fine-tuning pipeline + SGMSE.
`requirements-diff-erase.txt` is only needed for the optional AudioLDM
section further down -- skip that install if you're only debugging
`recipe=sgmse`.

In [ ]:
!pip install -q -r requirements-training.txt


## 4. Download the SGMSE checkpoint

Same checkpoint documented in `src/sgmse/VENDORED.md` (VoiceBank-DEMAND, 16kHz enhancement).

In [ ]:
import os
os.makedirs("checkpoints/sgmse", exist_ok=True)
!gdown 1_H3EXvhcYBhOZ9QNUcD5VZHc6ktrRbwQ -O "checkpoints/sgmse/train_vb_29nqe0uh_epoch=115.ckpt"


In [ ]:
# Sanity check: the checkpoint actually loads and runs a forward pass through SGMSEAttack.
import os
import sys
sys.path.insert(0, "src")
os.environ.setdefault("NO_TORCH_COMPILE", "1")

from audioseal_robust.attacks import SGMSEAttack
attack = SGMSEAttack(checkpoint="checkpoints/sgmse/train_vb_29nqe0uh_epoch=115.ckpt", num_steps=5)
print("Loaded OK:", type(attack._model).__name__)


## 5. (Optional, heavier) Download AudioLDM / DiffErase checkpoint

~10GB total (main checkpoint + VAE/CLAP/HiFiGAN support bundle) -- this is
why we do it here instead of the original dev box (only had 18GB free
total). Skip this section if you're only debugging `recipe=sgmse` right now.

Matches `src/audioldm_train/config/2023_08_23_reproduce_audioldm/audioldm_original.yaml`
(the vendored config `DiffEraseAttack` is set up to use) -- filenames below
are exactly what that config references by name, not arbitrary picks.

In [ ]:
#@title Download AudioLDM checkpoint + support bundle (~10GB, optional)
RUN_AUDIOLDM_DOWNLOAD = False  #@param {type:"boolean"}

if RUN_AUDIOLDM_DOWNLOAD:
    import os
    weights_root = "checkpoints/audioldm"
    ckpt_dir = os.path.join(weights_root, "data", "checkpoints")
    os.makedirs(ckpt_dir, exist_ok=True)

    # Main LatentDiffusion checkpoint (small architecture, matches audioldm_original.yaml)
    !wget -c -O "{ckpt_dir}/audioldm-s-full" "https://zenodo.org/records/7884686/files/audioldm-s-full?download=1"

    # Support bundle: VAE, AudioMAE, CLAP, HiFiGAN (16k + 48k) -- extracted filenames
    # are what audioldm_original.yaml references directly (vae_mel_16k_64bins.ckpt,
    # clap_htsat_tiny.pt, hifigan_16k_64bins.ckpt/.json).
    !wget -c -O "{weights_root}/checkpoints.tar" "https://zenodo.org/records/14342967/files/checkpoints.tar?download=1"
    !tar -xf "{weights_root}/checkpoints.tar" -C "{weights_root}"
    !rm "{weights_root}/checkpoints.tar"  # reclaim space once extracted

    !pip install -q -r requirements-diff-erase.txt
    print("AudioLDM checkpoint ready at:", os.path.join(ckpt_dir, "audioldm-s-full"))
else:
    print("Skipped -- set RUN_AUDIOLDM_DOWNLOAD = True to fetch AudioLDM weights.")


In [ ]:
#@title Sanity check: AudioLDM checkpoint loads through DiffEraseAttack (only if downloaded above)
if RUN_AUDIOLDM_DOWNLOAD:
    from audioseal_robust.attacks import DiffEraseAttack
    diff_erase_attack = DiffEraseAttack(
        checkpoint="checkpoints/audioldm/data/checkpoints/audioldm-s-full",
        config="src/audioldm_train/config/2023_08_23_reproduce_audioldm/audioldm_original.yaml",
        strength_max=0.08,
    )
    print("Loaded OK:", type(diff_erase_attack._model).__name__)


## 6. A tiny debug dataset

Not the real training data -- just enough distinct audio for the dataloader
to draw batches from, to confirm the training loop itself is correct. Swap
in a real `data.train_dir` for the actual run.

In [ ]:
import urllib.request, os

os.makedirs("debug_wavs", exist_ok=True)
url = "https://keithito.com/LJ-Speech-Dataset/LJ037-0171.wav"
for i in range(4):
    dest = f"debug_wavs/sample_{i}.wav"
    if not os.path.exists(dest):
        urllib.request.urlretrieve(url, dest)
print(os.listdir("debug_wavs"))


## 7. Debug training run -- `recipe=sgmse`

A handful of steps on GPU, not a real training budget. `attack.sgmse.num_steps`
is kept low here purely to keep the debug run fast -- raise it for the real run.

In [ ]:
!python -m audioseal_robust.train \
    recipe=sgmse \
    data.train_dir=debug_wavs \
    data.batch_size=2 data.num_workers=0 data.segment_duration=1.0 \
    attack.sgmse.checkpoint="checkpoints/sgmse/train_vb_29nqe0uh_epoch=115.ckpt" \
    attack.sgmse.num_steps=10 \
    epochs=1 updates_per_epoch=3 log_every=1 \
    checkpoint_dir=debug_checkpoints/sgmse \
    tracking.backend=none


## 8. Debug eval run -- `after_sgmse_training`

In [ ]:
!python -m audioseal_robust.evaluate \
    recipe=after_sgmse_training \
    eval_dir=debug_wavs \
    label=colab_debug \
    generator_checkpoint=debug_checkpoints/sgmse/generator_epoch0.pth \
    attack.sgmse.checkpoint="checkpoints/sgmse/train_vb_29nqe0uh_epoch=115.ckpt" \
    segment_duration=1.0 batch_size=2 n_eval_batches=1 \
    compute_pesq=false compute_sisnr=true \
    output_dir=debug_eval_out \
    tracking.backend=none


## 9. (Optional) Debug training run -- `recipe=diff_erase`

Only if you ran the AudioLDM download in step 5. Same idea -- a handful of
steps, not a real budget. This path backprops through a full VAE encode +
reverse-diffusion loop + VAE decode (see `DiffEraseAttack` docstring in
`src/audioseal_robust/attacks.py`), so it is meaningfully slower per step
than `recipe=sgmse` even on GPU -- expect this cell to take noticeably
longer than the sgmse one above for the same step count.

In [ ]:
if RUN_AUDIOLDM_DOWNLOAD:
    !python -m audioseal_robust.train \
        recipe=diff_erase \
        data.train_dir=debug_wavs \
        data.batch_size=2 data.num_workers=0 data.segment_duration=1.0 \
        attack.diff_erase.checkpoint="checkpoints/audioldm/data/checkpoints/audioldm-s-full" \
        attack.diff_erase.config="src/audioldm_train/config/2023_08_23_reproduce_audioldm/audioldm_original.yaml" \
        attack.diff_erase.strength_max=0.08 \
        epochs=1 updates_per_epoch=2 log_every=1 \
        checkpoint_dir=debug_checkpoints/diff_erase \
        tracking.backend=none
else:
    print("Skipped -- AudioLDM checkpoint wasn't downloaded (step 5).")


## 10. Next step

If every cell above ran clean: this recipe + these checkpoint paths are what
should go into the real training run. Hand off to the mentor to submit that
as a parallel run on proper compute -- don't scale the budget (`epochs`,
`updates_per_epoch`, `attack.sgmse.num_steps` / `attack.*.strength_max`) up
inside this notebook; that's what the main run is for.